# 02 — Estratégias de Chunking

## Por que chunking é um dos problemas mais importantes do RAG?

Na prática, chunking mal feito é responsável por boa parte das falhas de sistemas RAG.
Um LLM excelente com chunks ruins gera respostas ruins. Chunks bons com um LLM mediano ainda funcionam razoavelmente.

**O desafio:** dividir documentos em pedaços que sejam:
- Pequenos o suficiente para caber no contexto do LLM (tipicamente 512-2048 tokens)
- Grandes o suficiente para conter uma ideia **completa**
- Que não quebrem no meio de conceitos importantes
- Similares em tamanho (para evitar que um chunk domine os resultados)

**Estratégias que vamos comparar:**

| Estratégia | Complexidade | Quando usar |
|-----------|-------------|-------------|
| Fixed-size | Baixa | Protótipos, textos uniformes |
| Recursive split | Média | **Recomendado como padrão** |
| Semantic chunking | Alta | Documentos longos e heterogêneos |

In [ ]:
# Setup
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Texto de exemplo — artigo técnico sobre RAG
texto_exemplo = """
RAG (Retrieval-Augmented Generation) é uma técnica que combina recuperação de informação com geração de texto.

O processo de indexação converte documentos em embeddings vetoriais que são armazenados num banco de dados vetorial.
Durante a consulta, a query do usuário é convertida em embedding e os documentos mais similares são recuperados.

HNSW (Hierarchical Navigable Small World) é o algoritmo de indexação mais usado em bancos vetoriais.
Ele constrói um grafo hierárquico onde nós são conectados a vizinhos próximos em múltiplas camadas.
A busca começa na camada superior (poucos nós, conexões longas) e desce até encontrar os vizinhos mais próximos.

Embeddings são representações densas de texto em espaços de alta dimensão.
Modelos como all-MiniLM-L6-v2 geram vetores de 384 dimensões treinados para capturar semântica.
Textos semanticamente similares ficam próximos no espaço vetorial, permitindo busca por similaridade.

Chunking é o processo de dividir documentos longos em pedaços menores para indexação.
A escolha da estratégia de chunking afeta diretamente a qualidade do retrieval.
Chunks muito pequenos perdem contexto; chunks muito grandes são menos precisos na busca.
"""

print(f"Texto de exemplo: {len(texto_exemplo)} caracteres, {len(texto_exemplo.split())} palavras")

## 2.1 Fixed-Size Chunking

A estratégia mais simples: divide o texto a cada N caracteres, com overlap de M caracteres.

**Como funciona:** corta o texto mecanicamente, sem considerar estrutura.

**Overlap:** os últimos M caracteres do chunk anterior são repetidos no início do próximo.
Isso evita que informações que "cruzam" a fronteira se percam completamente.

**Vantagem:** simples, previsível, sem dependências extras.

**Desvantagem:** pode cortar no meio de uma frase ou ideia:
```
Chunk 1: "...O algoritmo HNSW usa uma estrutura hierárquica de"
Chunk 2: "grafos onde nós são conectados a vizinhos próximos..."
```
Chunk 1 termina no meio de uma frase — se recuperado isoladamente, perde o contexto.

In [ ]:
def fixed_size_chunk(text, chunk_size=500, overlap=50):
    """Divide texto em pedaços de tamanho fixo com overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks_fixed = fixed_size_chunk(texto_exemplo, chunk_size=300, overlap=50)

print(f"Fixed-size (300 chars, overlap=50): {len(chunks_fixed)} chunks")
for i, c in enumerate(chunks_fixed):
    print(f"  Chunk {i}: {len(c)} chars | '{c[:80].strip()}...'")

### Observação: onde corta?

Note que o fixed-size pode cortar no meio de uma frase.
O texto sobre HNSW provavelmente ficou dividido em pontos arbitrários — não em fronteiras semânticas.

Para textos com estrutura clara (parágrafos, seções), isso é problemático.
Para textos uniformes (transcrições de fala, logs), funciona razoavelmente.

## 2.2 Recursive Character Split

Uma evolução do fixed-size: em vez de cortar sempre no mesmo ponto, tenta **respeitar a estrutura** do texto.

**Como funciona:** define uma hierarquia de separadores:
1. `\n\n` (parágrafo) — divide aqui se possível
2. `\n` (linha) — se o chunk ainda for grande, divide por linha
3. `. ` (frase) — se ainda for grande, divide por frase
4. ` ` (palavra) — último recurso

O algoritmo aplica esses separadores recursivamente até que todos os chunks estejam abaixo do tamanho máximo.

**Resultado:** chunks que respeitam parágrafos e frases — muito mais coerentes semanticamente.

**Este é o padrão recomendado para a maioria dos casos.** LangChain e LlamaIndex usam isso como default.

In [ ]:
def recursive_chunk(text, chunk_size=500, overlap=50, separators=None):
    """Divide texto respeitando hierarquia de separadores."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    def split_text(text, separators):
        if not separators or len(text) <= chunk_size:
            return [text] if text.strip() else []

        sep = separators[0]
        parts = text.split(sep) if sep else list(text)

        chunks = []
        current = ""
        for part in parts:
            test = current + (sep if current else "") + part
            if len(test) <= chunk_size:
                current = test
            else:
                if current.strip():
                    chunks.append(current.strip())
                if len(part) > chunk_size:
                    # Parte ainda grande: recursão com próximo separador
                    sub_chunks = split_text(part, separators[1:])
                    chunks.extend(sub_chunks)
                    current = ""
                else:
                    current = part
        if current.strip():
            chunks.append(current.strip())
        return chunks

    raw_chunks = split_text(text, separators)

    # Adiciona overlap
    result = []
    for i, chunk in enumerate(raw_chunks):
        if i > 0 and overlap > 0:
            prev_end = raw_chunks[i-1][-overlap:]
            chunk = prev_end + " " + chunk
        result.append(chunk)
    return result

chunks_recursive = recursive_chunk(texto_exemplo, chunk_size=300, overlap=50)

print(f"Recursive split (300 chars, overlap=50): {len(chunks_recursive)} chunks")
for i, c in enumerate(chunks_recursive):
    print(f"  Chunk {i}: {len(c)} chars | '{c[:80].strip()}...'")

### Diferença chave em relação ao fixed-size

O recursive split tende a terminar chunks em pontos "naturais" do texto — fim de parágrafo, fim de frase.
Compare com o fixed-size: o conteúdo dos chunks é mais coerente e cada um tende a representar uma ideia completa.

Isso impacta diretamente o retrieval: se o chunk contém uma ideia completa, o embedding captura melhor seu significado.

## 2.3 Semantic Chunking

A estratégia mais sofisticada: divide onde o **significado muda**, não onde o texto tem quebras.

**Como funciona:**
1. Divide o texto em sentenças
2. Computa o embedding de cada sentença (ou janela de sentenças)
3. Calcula a similaridade entre sentenças consecutivas
4. Quando a similaridade cai abruptamente (mudança de tópico), cria um novo chunk

**Vantagem:** captura fronteiras temáticas reais — cada chunk trata de um assunto coeso.

**Custo:** precisa embedar todas as sentenças antes de chunkar. Para documentos longos, isso pode ser 3-5x mais lento que os outros métodos.

**Quando vale:** documentos longos e heterogêneos onde mudanças de tópico são frequentes e importantes.

In [ ]:
def semantic_chunk(text, model, threshold=0.7, min_chunk_size=100):
    """Divide texto em pontos de mudança semântica."""
    # Divide em sentenças
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    if len(sentences) <= 1:
        return [text]

    # Embeda cada sentença
    embs = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)

    # Calcula similaridade entre sentenças consecutivas
    similarities = []
    for i in range(len(sentences) - 1):
        sim = float(np.dot(embs[i], embs[i+1]))
        similarities.append(sim)

    # Identifica pontos de quebra (baixa similaridade = mudança de tópico)
    chunks = []
    current_chunk = sentences[0]

    for i, sim in enumerate(similarities):
        if sim < threshold and len(current_chunk) >= min_chunk_size:
            chunks.append(current_chunk.strip())
            current_chunk = sentences[i+1]
        else:
            current_chunk += " " + sentences[i+1]

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

chunks_semantic = semantic_chunk(texto_exemplo, model, threshold=0.65)

print(f"Semantic chunking (threshold=0.65): {len(chunks_semantic)} chunks")
for i, c in enumerate(chunks_semantic):
    print(f"  Chunk {i}: {len(c)} chars")
    print(f"    '{c[:100].strip()}...'")
    print()

### O que o semantic chunking encontrou?

Se funcionou bem, cada chunk deve corresponder a um dos tópicos do texto:
- Chunk sobre RAG em geral
- Chunk sobre HNSW
- Chunk sobre Embeddings
- Chunk sobre Chunking

Compare com os chunks fixed-size: os limites faziam sentido semântico aqui?

**Limitação importante:** o threshold (0.65) precisa ser ajustado para cada tipo de documento.
Um threshold muito alto = chunks minúsculos. Muito baixo = chunks enormes.

## 2.4 Comparação: Impacto na Qualidade de Retrieval

Métricas de texto não são suficientes — precisamos medir o impacto no que realmente importa: **o sistema encontra o chunk certo para cada query?**

In [ ]:
# Compara retrieval quality das 3 estratégias
queries_com_resposta = [
    ("Como funciona o HNSW?", "HNSW"),
    ("O que é um embedding?", "Embeddings"),
    ("Como o RAG funciona?", "RAG"),
    ("Por que chunking importa?", "Chunking"),
]

results = {}
for strategy_name, chunks in [
    ("Fixed-size", chunks_fixed),
    ("Recursive", chunks_recursive),
    ("Semantic", chunks_semantic),
]:
    chunk_embs = model.encode(chunks, normalize_embeddings=True, show_progress_bar=False)
    hits = 0
    for query, expected_topic in queries_com_resposta:
        q_emb = model.encode(query, normalize_embeddings=True)
        scores = chunk_embs @ q_emb
        best_chunk = chunks[np.argmax(scores)]
        # Verifica se o chunk mais relevante contém o tópico esperado
        if expected_topic.lower() in best_chunk.lower():
            hits += 1
    results[strategy_name] = hits / len(queries_com_resposta)
    print(f"{strategy_name}: {hits}/{len(queries_com_resposta)} queries com chunk correto como top-1")

print("
Recall@1 por estratégia:")
for name, recall in sorted(results.items(), key=lambda x: -x[1]):
    bar = "█" * int(recall * 20)
    print(f"  {name:12s}: {bar} {recall:.0%}")

### O que os números revelam?

A diferença entre estratégias mostra por que chunking merece atenção:
- **Fixed-size** frequentemente corta no meio de ideias, misturando tópicos num mesmo chunk
- **Recursive** respeita parágrafos, gerando chunks mais coesos — melhor recall
- **Semantic** encontra fronteiras de tópico reais, mas depende do threshold escolhido

**Insight chave:** invista tempo escolhendo a estratégia de chunking certa *antes* de otimizar o modelo de embedding ou o LLM. Um bom chunking com um modelo simples supera um modelo excelente com chunks ruins.

## Resumo

| Estratégia | Recall típico | Custo | Recomendação |
|-----------|--------------|-------|--------------|
| Fixed-size | Baixo-médio | O(1) | Só para protótipos |
| **Recursive split** | **Médio-alto** | **O(N)** | **Padrão recomendado** |
| Semantic | Alto | O(N×M) | Documentos longos e heterogêneos |

**Parâmetros para começar:**
- `chunk_size`: 256-512 caracteres (ou tokens)
- `overlap`: 10-20% do chunk_size (garante contexto nas fronteiras)

**Próximos passos:**
- [03 — Retrieval Strategies](03_retrieval_strategies.html): você tem bons chunks — como encontrá-los eficientemente?